In [5]:
import json
import time
import numpy as np
from pathlib import Path

In [6]:
PROJECT_DIR = Path("..")

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
EMBEDDINGS_DIR = DATA_DIR / "embeddings"

DOCUMENTS_FILE = PROCESSED_DIR / "documents.jsonl"
EMBEDDINGS_FILE = EMBEDDINGS_DIR / "embeddings.npy"
IDS_FILE = EMBEDDINGS_DIR / "ids.npy"

print("Documents:", DOCUMENTS_FILE)
print("Embeddings:", EMBEDDINGS_FILE)
print("IDs:", IDS_FILE)

Documents: ..\data\processed\documents.jsonl
Embeddings: ..\data\embeddings\embeddings.npy
IDs: ..\data\embeddings\ids.npy


In [7]:
documents = []

with open(DOCUMENTS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            documents.append(json.loads(line))

print("Documents loaded:", len(documents))
print(documents[0])

Documents loaded: 119921
{'id': 1, 'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling band of ultra-cynics, are seeing green again.", 'metadata': {'category': 2}}


In [8]:
embeddings = np.load(EMBEDDINGS_FILE)
ids = np.load(IDS_FILE)

print("Embeddings shape:", embeddings.shape)
print("IDs shape:", ids.shape)
print("Embedding dtype:", embeddings.dtype)

Embeddings shape: (119921, 384)
IDs shape: (119921,)
Embedding dtype: float32


In [9]:
class ExactIndex:
    def __init__(self, vectors, ids):
        self.vectors = vectors
        self.ids = ids

        print(f"Index created with {len(vectors)} vectors")
        print(f"Dimension: {vectors.shape[1]}")

    def search(self, query_vector, k=10):
        scores = self.vectors @ query_vector
        top_indices = np.argsort(scores)[::-1][:k]

        results = []

        for index in top_indices:
            results.append({
                "id": int(self.ids[index]),
                "score": float(scores[index]),
                "index": int(index),
            })

        return results

In [10]:
exact_index = ExactIndex(
    embeddings,
    ids
)

Index created with 119921 vectors
Dimension: 384


In [11]:
def cosine_similarity(query, vectors):
    return vectors @ query

In [12]:
exact_index = ExactIndex(
    embeddings,
    ids
)

Index created with 119921 vectors
Dimension: 384


In [13]:
query_index = 0

query_vector = embeddings[query_index]

results = exact_index.search(
    query_vector,
    k=5
)

for rank, result in enumerate(results, start=1):
    print(
        f"Rank {rank} | "
        f"ID: {result['id']} | "
        f"Score: {result['score']:.4f}"
    )

Rank 1 | ID: 1 | Score: 1.0000
Rank 2 | ID: 10 | Score: 0.9671
Rank 3 | ID: 3807 | Score: 0.5746
Rank 4 | ID: 8755 | Score: 0.5269
Rank 5 | ID: 10600 | Score: 0.5211


In [14]:
for rank, result in enumerate(results, start=1):
    index = result["index"]

    print(f"\n{'=' * 80}")
    print(f"Rank: {rank}")
    print(f"ID: {result['id']}")
    print(f"Similarity: {result['score']:.4f}")
    print(f"Text: {documents[index]['text'][:500]}")


Rank: 1
ID: 1
Similarity: 1.0000
Text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling band of ultra-cynics, are seeing green again.

Rank: 2
ID: 10
Similarity: 0.9671
Text: Wall St. Bears Claw Back Into the Black NEW YORK (Reuters) - Short-sellers, Wall Street's dwindling band of ultra-cynics, are seeing green again.

Rank: 3
ID: 3807
Similarity: 0.5746
Text: Wall St. Seen Lower on Oil; Google Eyed (Reuters) Reuters - Sky-high oil prices are likely to pressure Wall Street once again on Thursday, while earnings ews from tech giants Ciena (CIEN.N) and Nortel (NT.TO) and Google's (GOOG.O) awaited Nasdaq debut will also steer sentiment.

Rank: 4
ID: 8755
Similarity: 0.5269
Text: Wall St. Seen Rising as Oil Prices Slip (Reuters) Reuters - U.S. stock futures pointed toward a higher Wall Street open on Tuesday after crude oil prices fell for a third day, easing investors' fears that costly oil would squeeze company profits and slow growth.


In [15]:
start = time.perf_counter()

results = exact_index.search(
    query_vector,
    k=10
)

end = time.perf_counter()

search_time_ms = (end - start) * 1000

print(f"Search time: {search_time_ms:.3f} ms")

Search time: 16.058 ms


In [17]:
rng = np.random.default_rng(42)

random_query = rng.normal(
    size=embeddings.shape[1]
).astype(np.float32)

random_query /= np.linalg.norm(random_query)

results = exact_index.search(
    random_query,
    k=5
)

for rank, result in enumerate(results, start=1):
    print(
        f"Rank {rank} | "
        f"ID: {result['id']} | "
        f"Score: {result['score']:.4f}"
    )

Rank 1 | ID: 30702 | Score: 0.1994
Rank 2 | ID: 28578 | Score: 0.1986
Rank 3 | ID: 35541 | Score: 0.1849
Rank 4 | ID: 100070 | Score: 0.1750
Rank 5 | ID: 30953 | Score: 0.1743


In [18]:
query_indices = rng.choice(
    len(embeddings),
    size=100,
    replace=False
)

query_vectors = embeddings[query_indices]

times = []

for query in query_vectors:
    start = time.perf_counter()

    exact_index.search(query, k=10)

    end = time.perf_counter()

    times.append((end - start) * 1000)

times = np.array(times)

print(f"Queries: {len(times)}")
print(f"Average: {times.mean():.3f} ms")
print(f"Median: {np.median(times):.3f} ms")
print(f"Min: {times.min():.3f} ms")
print(f"Max: {times.max():.3f} ms")

Queries: 100
Average: 11.360 ms
Median: 10.500 ms
Min: 9.348 ms
Max: 42.220 ms


In [19]:
num_vectors = embeddings.shape[0]
dimension = embeddings.shape[1]
comparisons_per_query = num_vectors

print("Vectors:", num_vectors)
print("Dimensions:", dimension)
print("Vector comparisons per query:", comparisons_per_query)

Vectors: 119921
Dimensions: 384
Vector comparisons per query: 119921
